# 1. Extraire les données des destinations
* Objectif : Obtenir les coordonnées GPS des 35 villes
* Comment : API Nominatim API
* Données à extraire : CSV (ID_ville, ville, latitude, longitude)

(bibliothèque python également disponible)

In [1]:
#liste des 35 villes
villes =["Mont Saint Michel",
"St Malo",
"Bayeux",
"Le Havre",
"Rouen",
"Paris",
"Amiens",
"Lille",
"Strasbourg",
"Chateau du Haut Koenigsbourg",
"Colmar",
"Eguisheim",
"Besancon",
"Dijon",
"Annecy",
"Grenoble",
"Lyon",
"Gorges du Verdon",
"Bormes les Mimosas",
"Cassis",
"Marseille",
"Aix en Provence",
"Avignon",
"Uzes",
"Nimes",
"Aigues Mortes",
"Saintes Maries de la mer",
"Collioure",
"Carcassonne",
"Ariege",
"Toulouse",
"Montauban",
"Biarritz",
"Bayonne",
"La Rochelle"]

# liste pour stocker les résultats
coordonnées = []

In [2]:
# importer les bibliothèques nécessaires
import requests # pour faire des requêtes HTTP
import time # pour ajouter un délai entre les requêtes
import uuid # pour générer des identifiants uniques
import pandas as pd # pour manipuler les données
import csv # pour lire et écrire des fichiers CSV

# itérer sur chaque ville pour obtenir les coordonnées
for ville in villes:
    params = {
        'q': ville , # q=paramètre de recherche
        'format': 'json', #format=json pour obtenir la réponse en JSON
        'limit': 1 #pour obtenir une seule réponse par ville (car plusieurs réponses possibles)
    } 

#sans headers, la requête peut être bloquée par le serveur = code 403
    response = requests.get("https://nominatim.openstreetmap.org/search", params=params, headers={'User-Agent': 'Mozilla/5.0'})
        
    if response.status_code == 200: #si requête réussie
        data = response.json() #convertir la réponse en JSON
        if data: #si des données sont trouvées
            # créer un dictionnaire avec les coordonnées et la ville
            coord = { 
                'id': str(uuid.uuid4()),  # Générer un identifiant unique
                'ville': ville, #nom de la ville
                'latitude': data[0]['lat'], #latitude de la ville
                'longitude': data[0]['lon'] #longitude de la ville
                }
            coordonnées.append(coord)
        else:
            print(f"Aucune donnée trouvée pour {ville}") # si aucune donnée n'est trouvée
    else:
        print(f"Erreur lors de la requête pour {ville} :", response.status_code) #code d'erreur de la requête

    time.sleep(1)

In [3]:
#sauvegarder les résultats dans un fichier csv
df=pd.DataFrame(coordonnées)
df.to_csv("coordonnées_villes.csv", index=False, encoding='utf-8') #index=False, ne pas inclure l'index dans le fichier CSV
print("Les coordonnées des villes ont été enregistrées dans 'coordonnées_villes.csv'.")

Les coordonnées des villes ont été enregistrées dans 'coordonnées_villes.csv'.


In [4]:
#ouvrir le fichier csv et afficher son contenu
with open('coordonnées_villes.csv', mode='r', encoding='utf-8') as file: # ouvrir le fichier en mode lecture
    reader = csv.reader(file)
    for row in reader:
        print(row)

['id', 'ville', 'latitude', 'longitude']
['dee7f8f0-050c-4868-aa4f-290b89b631f3', 'Mont Saint Michel', '48.6359541', '-1.5114600']
['962153af-2f03-458f-ad3e-0949042e979c', 'St Malo', '49.3146950', '-96.9538228']
['465c510f-bbf6-4241-b25c-d817701d6d5e', 'Bayeux', '49.2764624', '-0.7024738']
['02f1bad4-03d3-477f-8995-97324e68923f', 'Le Havre', '49.4938975', '0.1079732']
['74a1fde5-c83c-48bd-ad6a-0f6bcd67a5a7', 'Rouen', '49.4404591', '1.0939658']
['38416851-7961-421b-a69f-3edf8bb09c0f', 'Paris', '48.8588897', '2.3200410']
['2872d751-99ee-4f29-9301-c9aeec503b0d', 'Amiens', '49.8941708', '2.2956951']
['9f22dfd2-eeb6-41c5-8efe-e11ba8756099', 'Lille', '50.6365654', '3.0635282']
['6dd738a9-928f-43df-be0a-41e1b889a4ce', 'Strasbourg', '48.5846140', '7.7507127']
['aebca5b2-35b4-48c2-902a-9c8430213e07', 'Chateau du Haut Koenigsbourg', '48.2494107', '7.3443202']
['7eac4c3c-cce4-4eb2-81b3-b7090253642e', 'Colmar', '48.0777517', '7.3579641']
['34c4c6cc-c681-4930-9cff-82185339339c', 'Eguisheim', '48.04

# 2. Collecte des données météo (prévisions +7 jours)
* Objectif : Obtenir les prévisions météo des 35 villes
* Comment : 
    * OpenWeatherMap - One Call API 
    * Critères : températures, pluie, humidité, qualité de l'air
* Sortie : 

In [19]:
#clé API OpenWeatherMap
API_KEY = '29e9493ff0392fd7df6e77036ab94b5a'

#test pour vérifier si la clé API est valide
response = requests.get(f'https://api.openweathermap.org/data/3.0/onecall?lat=46.1597320&lon=-1.1515951&exclude=minutely,hourly,alerts&appid={API_KEY}')

print(response.status_code) # afficher le code de statut de la réponse

401


In [20]:

#créer les colonnes météo que je veux ajouter à mon fichier CSV
df['summary'] = None #summary
df['temp_jour'] = None #daily.temp
df['ressenti'] = None #laily.feels_like
df['humidite'] = None #daily.humidity
df['pluie_7j'] = None #daily.rain
df['prob_pluie'] = None #daily.pop
df['indice_uv'] = None #daily.uvi
df['sunrise'] = None #daily.sunrise
df['sunset'] = None #daily.sunset

print(df.columns)

Index(['id', 'ville', 'latitude', 'longitude', 'summary', 'temp_jour',
       'ressenti', 'humidite', 'pluie_7j', 'prob_pluie', 'indice_uv',
       'sunrise', 'sunset'],
      dtype='object')


In [ ]:
for i, row in df.iterrows():
    lat = row['latitude']
    lon = row['longitude']

    #paramètres pour la requête API OpenWeatherMap
    params = {
        'lat': lat,
        'lon': lon,
        'exclude': 'minutely,hourly,alerts', # Exclure les données non nécessaires
        'units': 'metric', # Unités métriques
        'appid': API_KEY # Clé API OpenWeatherMap
    }

    response = requests.get("https://api.openweathermap.org/data/3.0/Onecall", params=params, headers={'User-Agent': 'Mozilla/5.0'})

    if response.status_code == 200:
        data = response.json()

        # Données sur 4 jours
        daily = data.get('daily', [])

        if daily:
            # Moyenne température sur 7 jours
            moy_temp = sum(jour['temp']['day'] for jour in daily[:7]) / 7
            humidite = sum(jour['humidity'] for jour in daily[:7]) / 7
            pluie = sum(jour.get('rain', 0) for jour in daily[:7])  # parfois pas de clé 'rain'

            df.at[i, 'temp_moyenne'] = round(moy_temp, 1)
            df.at[i, 'humidite'] = round(humidite, 1)
            df.at[i, 'pluie_7j'] = round(pluie, 1)

            print(f"Météo ajoutée pour {row['ville']}")
        else:
            print(f"Pas de données météo pour {row['ville']}")
    else:
        print(f"Erreur API pour {row['ville']}: {response.status_code}")

    time.sleep(1)  # Pause pour respecter les quotas de l’API

# Enregistrement du fichier enrichi
df.to_csv("villes_meteo.csv", index=False)
print(" Fichier 'villes_meteo.csv' exporté avec météo.")


Pas de données météo pour Mont Saint Michel
Pas de données météo pour St Malo
Pas de données météo pour Bayeux
Pas de données météo pour Le Havre
Pas de données météo pour Rouen
Pas de données météo pour Paris


KeyboardInterrupt: 

In [9]:
#ouvrir le fichier csv et afficher son contenu
with open('villes_meteo.csv', mode='r', encoding='utf-8') as file: # ouvrir le fichier en mode lecture
    reader = csv.reader(file)
    for row in reader:
        print(row)

['id', 'ville', 'latitude', 'longitude', 'summary', 'temp_jour', 'ressenti', 'humidite', 'pluie_7j', 'prob_pluie', 'indice_uv']
['ebeeb0ce-c337-4567-b48d-759dd86f1b79', 'Mont Saint Michel', '48.6359541', '-1.5114600', '', '', '', '', '', '', '']
['87dfdae5-71d5-4ba3-b17c-00a445a38529', 'St Malo', '49.3146950', '-96.9538228', '', '', '', '', '', '', '']
['ed123c46-8cbc-403f-aa0a-b417a39528aa', 'Bayeux', '49.2764624', '-0.7024738', '', '', '', '', '', '', '']
['94c00fc2-6b5b-4d41-b0da-1aac87158045', 'Le Havre', '49.4938975', '0.1079732', '', '', '', '', '', '', '']
['1bae3aa3-1c4a-4bb3-9b83-8caa9ec4e74d', 'Rouen', '49.4404591', '1.0939658', '', '', '', '', '', '', '']
['379f55e7-7048-4a47-ba86-f1aea6039a91', 'Paris', '48.8534951', '2.3483915', '', '', '', '', '', '', '']
['9240139c-526b-4361-9185-d43b6cc1e4a9', 'Amiens', '49.8941708', '2.2956951', '', '', '', '', '', '', '']
['3564ed96-c55b-43c0-8148-88f3c0b235c0', 'Lille', '50.6365654', '3.0635282', '', '', '', '', '', '', '']
['a87e4bc

# 3. Scraping Booking.com
* Quoi : récupérer les informations d'hôtels pour chaque ville
* Comment : scrapy
* Données à extraire : CSV (ID_hôtel, city_ID, nom_hôtel, url, latitude, longitude, note, description)

In [10]:
'''from bs4 import BeautifulSoup
import urllib.parse # pour encoder les URL (gérer les espaces et caractères spéciaux)

ville = "Paris"  # ville recherchée, par exemple "Paris"
url_ville = urllib.parse.quote_plus(ville) 

url = f"https://www.booking.com/searchresults.fr.html?ss={url_ville}"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36',
    'Accept-Language': 'fr-FR,fr;q=0.9,en-US;q=0.8,en;q=0.7',
    'Referer': 'https://www.google.com/'}

response = requests.get(url, headers=headers)
hotels_data = []

print(response.status_code)
print(response.text[:500])  # affiche les 500 premiers caractères


if response.status_code == 200:
    soup = BeautifulSoup(response.content, 'html.parser')
    hotels = soup.select('[data-testid="property-card"]')[:10]  # Limiter à 10 hôtels

    for hotel in hotels:
        try:
            name = hotel.select_one('[data-testid="title"]').get_text(strip=True)
        except:
            name = "N/A"

        try:
            link_suffix = hotel.select_one('[data-testid="title"] a')['href']
            link = "https://www.booking.com" + link_suffix
        except:
            link = "N/A"

        try:
            rating = hotel.select_one('[data-testid="review-score"]').get_text(strip=True)
        except:
            rating = "N/A"

        try:
            description = hotel.select_one('[data-testid="review-score-component"]').get_text(strip=True)
        except:
            description = "N/A"

        # Booking ne fournit pas toujours les coordonnées GPS sur la page de recherche
        latitude = "N/A"
        longitude = "N/A"

        hotels_data.append({
            'ville': ville,
            'name': name,
            'link': link,
            'latitude': latitude,
            'longitude': longitude,
            'rating': rating,
            'description': description
        })

    print("Scraping terminé pour", ville)
else:
    print(f"Erreur HTTP {response.status_code} pour {ville}")

print(response.status_code)
print(response.text[:500])  # affiche les 500 premiers caractères

print(f"Nombre d'hôtels trouvés pour {ville} :", len(hotels))'''

'from bs4 import BeautifulSoup\nimport urllib.parse # pour encoder les URL (gérer les espaces et caractères spéciaux)\n\nville = "Paris"  # ville recherchée, par exemple "Paris"\nurl_ville = urllib.parse.quote_plus(ville) \n\nurl = f"https://www.booking.com/searchresults.fr.html?ss={url_ville}"\nheaders = {\'User-Agent\': \'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36\',\n    \'Accept-Language\': \'fr-FR,fr;q=0.9,en-US;q=0.8,en;q=0.7\',\n    \'Referer\': \'https://www.google.com/\'}\n\nresponse = requests.get(url, headers=headers)\nhotels_data = []\n\nprint(response.status_code)\nprint(response.text[:500])  # affiche les 500 premiers caractères\n\n\nif response.status_code == 200:\n    soup = BeautifulSoup(response.content, \'html.parser\')\n    hotels = soup.select(\'[data-testid="property-card"]\')[:10]  # Limiter à 10 hôtels\n\n    for hotel in hotels:\n        try:\n            name = hotel.select_one(\'[data-testid

In [24]:
%pip install websocket-client


Note: you may need to restart the kernel to use updated packages.


In [25]:
import websockets
import urllib.parse # pour encoder les URL (gérer les espaces et caractères spéciaux)
import time # pour ajouter un délai entre les requêtes
from selenium import webdriver # pour automatiser le navigateur
from selenium.webdriver.chrome.options import Options # pour configurer les options du navigateur (ici, Chrome)
from selenium.webdriver.common.by import By # pour sélectionner des éléments dans la page

options = Options() 
options.add_argument("--headless") # exécuter Chrome en mode sans tête (pas d'interface graphique)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36") # pour simuler un navigateur réel

driver = webdriver.Chrome(options=options) # pour initialiser le navigateur Chrome avec les options définies


for ville in villes: # itérer sur chaque ville
    print(f"Recherche d'hôtels à {ville}...") # afficher la ville en cours de recherche
    # encoder la ville pour l'URL
    ville_encoded = urllib.parse.quote_plus(ville)
    url = f"https://www.booking.com/searchresults.fr.html?ss={ville_encoded}" # encoder la ville dans l'URL
    driver.get(url) # ouvrir l'URL dans le navigateur
    time.sleep(5)  # attendre que la page charge

    hotels_data = [] # liste pour stocker les données des hôtels

    hotels = driver.find_elements(By.CSS_SELECTOR, '[data-testid="property-card"]')[:5]

    for hotel in hotels:
        try:
            name = hotel.find_element(By.CSS_SELECTOR, '[data-testid="title"]').text
        except:
            name = "N/A"

        try:
            rating = hotel.find_element(By.CSS_SELECTOR, '[data-testid="review-score"]').text
        except:
            rating = "N/A"
        try:
            price = hotel.find_element(By.CSS_SELECTOR, '[data-testid="price-and-discounted-price"]').text
        except:
            price = "N/A"

        print(f"Hôtel: {name} | Note: {rating} | Prix: {price}")

driver.quit()


Recherche d'hôtels à Mont Saint Michel...
Hôtel: Mercure Mont Saint Michel | Note: Avec une note de 8,3
8,3
Très bien
3 840 expériences vécues | Prix: N/A
Hôtel: Auberge Saint Pierre | Note: Avec une note de 8,2
8,2
Très bien
1 214 expériences vécues | Prix: N/A
Hôtel: Les Terrasses Poulard | Note: Avec une note de 7,3
7,3
Bien 
3 644 expériences vécues | Prix: N/A
Hôtel: La Mère Poulard | Note: Avec une note de 7,6
7,6
Bien 
3 419 expériences vécues | Prix: N/A
Hôtel: La Vieille Auberge | Note: Avec une note de 7,5
7,5
Bien 
1 565 expériences vécues | Prix: N/A
Recherche d'hôtels à St Malo...
Hôtel:  | Note:  | Prix: N/A
Hôtel:  | Note:  | Prix: N/A
Hôtel:  | Note:  | Prix: N/A
Hôtel:  | Note:  | Prix: N/A
Hôtel:  | Note:  | Prix: N/A
Recherche d'hôtels à Bayeux...
Hôtel: Hotel Le Lion D'Or et Restaurant La Table Du Lion | Note: Avec une note de 8,6
8,6
Superbe
1 125 expériences vécues | Prix: N/A
Hôtel: Hôtel De Brunville & Spa | Note: Avec une note de 8,0
8,0
Très bien
1 860 expérie

In [ ]:
# Enregistrement du fichier enrichi
df_hotels = pd.DataFrame(hotels)
df_hotels.to_csv("hotels_booking.csv", index=False, encoding='utf-8')

# 4. Création du Data Lake (S3)
* Quoi : Stocker tous les fichiers CSV dans des buckets S3
* Contenu à stocker : Données météo par ville / Liste des villes avec GPS / Données des hôtels
* Nom du bucket S3 : kayak-data-lake

# 5. ETL vers un entrepôt SQL
* Outils : AWS RDS (MySQL ou PostgreSQL)
* Étapes : Créer des tables SQL pour villes, meteo, hotels
* Charger les fichiers CSV depuis S3 dans la base de données avec un script ETL (Python, Airflow ou AWS Glue)
* Vérifier que les données sont bien normalisées (clé étrangère entre weather et cities, entre hotels et cities)

# 6. Visualisations
Outil recommandé : Plotly
Cartes à produire :
* Top 5 des villes avec la meilleure météo (selon ton indice)
* Top 20 hôtels (note utilisateur + météo favorable)

Représentation : cartes interactives avec clusters ou bulles



# Livrables finaux
* CSV enrichi → Stocké sur S3
* Base SQL sur AWS RDS contenant toutes les données
* Deux cartes Plotly :
    * Top 5 des destinations
    * Top 20 hôtels
* (Optionnel) : Documentation sur les critères météo utilisés + scripts utilisés